In [ ]:
!pip install -q "transformers>=4.40" "datasets>=2.19" "accelerate>=0.30" sentencepiece

In [ ]:
import os, random
import numpy as np
import pandas as pd
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import Dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_CUDA = torch.cuda.is_available()

OPTIONS = ["A", "B", "C", "D", "E"]
LABEL2ID = {L: i for i, L in enumerate(OPTIONS)}
ID2LABEL = {i: L for L, i in LABEL2ID.items()}

# Point these at course-provided fine-tuned checkpoints if you have them;
# otherwise the base models are fine-tuned below into 5-class checkpoints.
DEBERTA_CKPT = "microsoft/deberta-v3-small"
ROBERTA_CKPT = "roberta-base"
W_DEBERTA, W_ROBERTA = 0.70, 0.30
print("device:", DEVICE)

In [ ]:
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
train_full = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv").reset_index(drop=True)

def format_input(row):
    opts = " ".join(f"({L}) {row[L]}" for L in OPTIONS)
    return f"{row['prompt']} {opts}"

train_full = train_full.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
n_val = 200
val_df = train_full.iloc[:n_val].reset_index(drop=True)
train_df = train_full.iloc[n_val:].reset_index(drop=True)

train_texts = [format_input(r) for _, r in train_df.iterrows()]
train_labels = [LABEL2ID[a] for a in train_df["answer"]]
test_texts = [format_input(r) for _, r in test.iterrows()]
print("train:", len(train_df), " val:", len(val_df), " test:", len(test))

In [ ]:
def finetune(model_name, texts, labels, epochs=3, bs=16, lr=2e-5):
    tok = AutoTokenizer.from_pretrained(model_name)
    ds = Dataset.from_dict({"text": texts, "labels": labels}).map(
        lambda b: tok(b["text"], truncation=True, max_length=256),
        batched=True, remove_columns=["text"])
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5).to(DEVICE)
    args = TrainingArguments(
        output_dir=f"/kaggle/working/{model_name.split('/')[-1]}_m5",
        num_train_epochs=epochs, per_device_train_batch_size=bs, learning_rate=lr,
        fp16=False, report_to="none", seed=SEED, logging_steps=50, save_strategy="no")
    Trainer(model=model, args=args, train_dataset=ds, processing_class=tok,
            data_collator=DataCollatorWithPadding(tok)).train()
    return model.eval(), tok

deb_model, deb_tok = finetune(DEBERTA_CKPT, train_texts, train_labels)

In [ ]:
rob_model, rob_tok = finetune(ROBERTA_CKPT, train_texts, train_labels)

In [ ]:
@torch.no_grad()
def predict_probs(texts, model, tokenizer, bs=32, prefix=None):
    model.eval()
    if prefix:
        texts = [f"{prefix} {t}" for t in texts]
    out = []
    for i in range(0, len(texts), bs):
        enc = tokenizer(texts[i:i + bs], truncation=True, max_length=256,
                        padding=True, return_tensors="pt").to(DEVICE)
        probs = torch.softmax(model(**enc).logits, dim=-1)
        out.append(probs.cpu().numpy())
    return np.concatenate(out, axis=0)

def top3_letters(prob_row):
    return [OPTIONS[j] for j in np.argsort(-prob_row)[:3]]

In [ ]:
ROW = 25
p_deb = predict_probs([test_texts[ROW]], deb_model, deb_tok)[0]
i = int(np.argmax(p_deb))
q1 = f"{OPTIONS[i]}, {p_deb[i]:.4f}"
print("Q1:", q1)

In [ ]:
p_rob = predict_probs([test_texts[ROW]], rob_model, rob_tok)[0]
p_avg = (p_deb + p_rob) / 2
q2 = OPTIONS[int(np.argmax(p_avg))]
print("Q2:", q2)

In [ ]:
p_wtd = W_DEBERTA * p_deb + W_ROBERTA * p_rob
q3 = OPTIONS[int(np.argmax(p_wtd))]
print("Q3:", q3)

In [ ]:
q4 = " ".join(top3_letters(p_wtd))
print("Q4:", q4)

In [ ]:
P_deb_test = predict_probs(test_texts, deb_model, deb_tok)
P_rob_test = predict_probs(test_texts, rob_model, rob_tok)
W_test = W_DEBERTA * P_deb_test + W_ROBERTA * P_rob_test

preds = [" ".join(top3_letters(W_test[i])) for i in range(len(test))]
submission = pd.DataFrame({"id": test["id"], "prediction": preds})
submission.to_csv("submission.csv", index=False)
q5 = len(submission)
print("Q5:", q5)

In [ ]:
N = 50
orig = P_deb_test[:N]
aug = predict_probs(test_texts[:N], deb_model, deb_tok,
                    prefix="Answer the following multiple-choice question carefully:")
tta = (orig + aug) / 2
q6 = int(sum(np.argmax(orig[i]) != np.argmax(tta[i]) for i in range(N)))
print("Q6:", q6)

In [ ]:
N = 100
q7 = int(sum(np.argmax(P_deb_test[i]) != np.argmax(W_test[i]) for i in range(N)))
print("Q7:", q7)

In [ ]:
N = 100
gain = W_test[:N].max(axis=1) - P_deb_test[:N].max(axis=1)
q8 = int((gain > 0).sum())
print("Q8:", q8)

In [ ]:
N = 100
q9 = int(sum(top3_letters(P_deb_test[i]) != top3_letters(W_test[i]) for i in range(N)))
print("Q9:", q9)

In [ ]:
def apk(actual, predicted, k=3):
    for i, p in enumerate(predicted[:k]):
        if p == actual:
            return 1.0 / (i + 1)
    return 0.0

val_texts = [format_input(r) for _, r in val_df.iterrows()]
Pd_val = predict_probs(val_texts[:100], deb_model, deb_tok)
Pr_val = predict_probs(val_texts[:100], rob_model, rob_tok)
W_val = W_DEBERTA * Pd_val + W_ROBERTA * Pr_val
val_ans = val_df["answer"].tolist()[:100]
q10 = round(float(np.mean([apk(val_ans[i], top3_letters(W_val[i])) for i in range(100)])), 4)
print("Q10:", q10)